In [ ]:
!git clone https://github.com/aria-yang/youtube-thumbnail-performance-predictor.git
%cd youtube-thumbnail-performance-predictor
!git checkout ablation

Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q numpy pandas pillow tqdm scikit-learn loguru typer python-dotenv
!pip install -q easyocr facenet-pytorch deepface

Run full merged pipeline

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive', force_remount=True)
REPO_ROOT = Path('/content/youtube-thumbnail-performance-predictor')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts')
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)
print(f'Repo root: {REPO_ROOT}')
print(f'Artifact root: {ARTIFACT_ROOT}')

In [ ]:
import shutil
from pathlib import Path

REPO_ROOT = Path('/content/youtube-thumbnail-performance-predictor')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts')

SYNC_PLAN = {
    REPO_ROOT / 'data' / 'processed': [
        'merged_cnn_cache_resnet50.csv',
        'merged_cnn_embeddings_resnet50.npy',
        'merged_labeled_data.csv',
        'merged_ocr_features.csv',
        'merged_text_embeddings.npy',
        'merged_face_cache.csv',
        'merged_face_embeddings.npy',
        'text_embeddings.npy',
        'face_embeddings.npy',
    ],
    REPO_ROOT / 'data' / 'splits': [
        'random_train.csv', 'random_val.csv', 'random_test.csv',
        'channel_train.csv', 'channel_val.csv', 'channel_test.csv',
        'time_train.csv', 'time_val.csv', 'time_test.csv',
    ],
    REPO_ROOT / 'models': [
        'fusion_mlp.pt',
        'fusion_mlp_shap.pt',
        'fusion_end_to_end_resnet50_head_only_proj_lr5e4.pt',
    ],
}

missing = []
for local_dir, filenames in SYNC_PLAN.items():
    local_dir.mkdir(parents=True, exist_ok=True)
    for filename in filenames:
        src_path = ARTIFACT_ROOT / filename
        dst_path = local_dir / filename
        if src_path.exists():
            shutil.copy2(src_path, dst_path)
            print(f'Imported {filename} -> {dst_path.relative_to(REPO_ROOT)}')
        else:
            missing.append(filename)
            print(f'Missing in Drive artifacts: {filename}')

if missing:
    print('Missing files are okay if this run will generate them.')
else:
    print('All requested artifacts imported from Drive.')

In [ ]:
%%bash
python training/train_fusion.py \
  --device cuda \
  --split_name random \
  --num_epochs 40 \
  --lr 1e-3 \
  --early_stopping_metric auroc \
  --early_stopping_patience 12 \
  --seed 42


## Tuning

In [ ]:
%%bash
python training/finetune_fusion_end_to_end.py \
  --device cuda \
  --split_name random \
  --batch_size 32 \
  --num_epochs 20 \
  --freeze_epochs 5 \
  --unfreeze_all_epoch 5 \
  --head_lr 5e-4 \
  --backbone_lr 1e-5 \
  --weight_decay 1e-4 \
  --hidden1 512 \
  --hidden2 256 \
  --dropout_p 0.35 \
  --checkpoint_path models/fusion_end_to_end_resnet50_head_only_proj_lr5e4.pt \
  --history_path data/processed/fusion_end_to_end_head_only_proj_lr5e4_history.csv


# Ablation and SHAP

In [ ]:
%%bash
python -m training.ablation_study


In [ ]:
%%bash
python -m training.run_shap_analysis


# Cross-Validation

In [ ]:
%%bash
python training/eval_crosssplit.py


# GitHub and Google Drive

In [ ]:
!git config user.name "aria-yang"
!git config user.email "ariayang1205@gmail.com"

In [ ]:
print('Skipping git push in Colab. Sync artifacts to Drive instead, then push code changes from your local machine.')

In [ ]:
import shutil
from pathlib import Path

REPO_ROOT = Path('/content/youtube-thumbnail-performance-predictor')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts')
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

export_files = [
    REPO_ROOT / 'data' / 'processed' / 'merged_labeled_data.csv',
    REPO_ROOT / 'data' / 'processed' / 'merged_cnn_embeddings_resnet50.npy',
    REPO_ROOT / 'data' / 'processed' / 'merged_cnn_cache_resnet50.csv',
    REPO_ROOT / 'data' / 'processed' / 'merged_ocr_features.csv',
    REPO_ROOT / 'data' / 'processed' / 'merged_text_embeddings.npy',
    REPO_ROOT / 'data' / 'processed' / 'merged_face_embeddings.npy',
    REPO_ROOT / 'data' / 'processed' / 'merged_face_cache.csv',
    REPO_ROOT / 'data' / 'splits' / 'random_train.csv',
    REPO_ROOT / 'data' / 'splits' / 'random_val.csv',
    REPO_ROOT / 'data' / 'splits' / 'random_test.csv',
    REPO_ROOT / 'data' / 'splits' / 'channel_train.csv',
    REPO_ROOT / 'data' / 'splits' / 'channel_val.csv',
    REPO_ROOT / 'data' / 'splits' / 'channel_test.csv',
    REPO_ROOT / 'data' / 'splits' / 'time_train.csv',
    REPO_ROOT / 'data' / 'splits' / 'time_val.csv',
    REPO_ROOT / 'data' / 'splits' / 'time_test.csv',
    REPO_ROOT / 'models' / 'fusion_mlp.pt',
    REPO_ROOT / 'models' / 'fusion_mlp_shap.pt',
    REPO_ROOT / 'models' / 'fusion_end_to_end_resnet50_head_only_proj_lr5e4.pt',
    REPO_ROOT / 'outputs' / 'cross_split_generalization.csv',
    REPO_ROOT / 'outputs' / 'cross_split_generalization.json',
    REPO_ROOT / 'outputs' / 'cross_split_auroc_f1.png',
    REPO_ROOT / 'outputs' / 'shap_feature_importance.csv',
    REPO_ROOT / 'outputs' / 'shap_top10_features.csv',
    REPO_ROOT / 'outputs' / 'shap_global_importance.png',
    REPO_ROOT / 'outputs' / 'shap_notes.md',
    REPO_ROOT / 'outputs' / 'ablation_plot.png',
    REPO_ROOT / 'outputs' / 'ablation_table.csv',
]

for src_path in export_files:
    if src_path.exists():
        dst_path = ARTIFACT_ROOT / src_path.name
        shutil.copy2(src_path, dst_path)
        print(f'Exported {src_path.relative_to(REPO_ROOT)} -> {dst_path}')
    else:
        print(f'Skipped missing file: {src_path.relative_to(REPO_ROOT)}')

In [ ]:
print('No git rewrite/push step in Colab. Use Drive as the artifact store, and update .gitignore locally in the repo when needed.')

In [ ]:
import os

# Ensure we are in the correct directory
if os.getcwd() != '/content/youtube-thumbnail-performance-predictor':
  %cd /content/youtube-thumbnail-performance-predictor

# Show commits that are local but not yet on the remote 'ablation' branch
!git log --oneline --graph origin/ablation..ablation